# Actividad 1: Carga, Transformación y Reporte de Goodreads Books

En este notebook se ejecuta el flujo analítico completo:
1. Configuración de rutas de importación hacia la carpeta `src/`.
2. Carga del dataset público `pfaha/goodreads-books` mediante `HFDatasetLoader`.
3. Aplicación de 4 transformaciones de datos orientadas al dominio literario. Se conservan las variables que usa la necesidad analítica: título, autor, páginas, calificación, editorial, idioma, número de calificaciones, número de reseñas, pertenencia a una serie y formato.
4. Generación e impresión del reporte estructurado y exportación a `reportes/goodreads_books.txt`.

In [1]:
import sys
from pathlib import Path

# 1. Configurar ruta dinámica hacia la carpeta 'src'
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

# 2. Importar las clases reutilizables del proyecto
from hf_loader import HFDatasetLoader
from reportes import Reportes

print(" Módulos cargados correctamente desde src/")

 Módulos cargados correctamente desde src/


In [2]:
# Instanciar la clase de carga con el dataset de Goodreads
loader = HFDatasetLoader(
    dataset_name="pfaha/goodreads-books",
    split="train"
)

# Cargar el DataFrame original de Pandas
df_original = loader.load()

print(f" Dataset cargado exitosamente.")
print(f"Filas: {df_original.shape[0]} | Columnas: {df_original.shape[1]}\n")
df_original.head(3)

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

 Dataset cargado exitosamente.
Filas: 500000 | Columnas: 24



,title,contributors,first_author,description,genres,series,format,pages,language,asin,...,average_rating,ratings_count,reviews_count,work_id,catalog_guideline_violation,book_id,url,num_editions,has_more_editions,is_part_of_series
0,La doctrina secreta. Vol I. Cosmogenesis (Cien...,"[{'name': 'Helena Petrovna Blavatsky', 'url': ...",Helena Petrovna Blavatsky,"Esta es una obra inteligente, original e instr...","[Occult, Spirituality, Nonfiction, Philosophy,...","[{'name': 'The Secret Doctrine', 'position': '...",Paperback,319.0,Spanish; Castilian,950171103X,...,4.16,255.0,22.0,44066165.0,False,7282453,https://www.goodreads.com/book/show/7282453,124.0,True,True
1,Clement Marot and the Inflections of Poetic Voice,"[{'name': 'Robert Griffin', 'url': 'https://ww...",Robert Griffin,This title is part of UC Press's Voices Revive...,[],[],Hardcover,NaN,English,0520025865,...,0.00,0.0,0.0,1320719.0,False,1331224,https://www.goodreads.com/book/show/1331224,2.0,True,False
2,How to Market Legal Services,"[{'name': 'Robert W Denney', 'url': 'https://w...",Robert W Denney,None,[],[],Hardcover,278.0,English,0442219806,...,0.00,0.0,0.0,3956063.0,False,3910678,https://www.goodreads.com/book/show/3910678,2.0,True,False


In [3]:
import pandas as pd

# Crear copia de trabajo desde los datos originales
df_transformado = df_original.copy()

# Transformación 1: Selección de columnas clave
# Se conservan también las variables de participación de lectores, series y formato
# que usa la necesidad analítica (secciones 2.4, 3 y 4.2 del documento)
columnas_clave = [
    'title', 'first_author', 'pages', 'average_rating', 'publisher', 'language',
    'ratings_count', 'reviews_count', 'is_part_of_series', 'format'
]
columnas_presentes = [col for col in columnas_clave if col in df_transformado.columns]
df_transformado = df_transformado[columnas_presentes].copy()

# Convertir cualquier lista o arreglo interno a texto para evitar el error 'unhashable type'
for col in df_transformado.columns:
    df_transformado[col] = df_transformado[col].apply(
        lambda x: ", ".join(map(str, x)) if isinstance(x, (list, tuple)) else x
    )

# Transformación 2: Renombrado de columnas a español
nombres_espanol = {
    'title': 'titulo',
    'first_author': 'autor',
    'pages': 'paginas',
    'average_rating': 'calificacion',
    'publisher': 'editorial',
    'language': 'idioma',
    'ratings_count': 'num_calificaciones',
    'reviews_count': 'num_resenas',
    'is_part_of_series': 'es_parte_de_serie',
    'format': 'formato'
}
df_transformado = df_transformado.rename(columns=nombres_espanol)

# Transformación 3: Limpieza de filas con valores nulos
# El formato es una variable de apoyo: sus nulos se etiquetan en lugar de eliminar la fila
if 'formato' in df_transformado.columns:
    df_transformado['formato'] = df_transformado['formato'].fillna('Sin informar')
df_transformado = df_transformado.dropna()

# Transformación 4: Filtrado por condición (Calificación >= 3.5)
if 'calificacion' in df_transformado.columns:
    df_transformado['calificacion'] = pd.to_numeric(df_transformado['calificacion'], errors='coerce')
    df_transformado = df_transformado[df_transformado['calificacion'] >= 3.5]

print(f"Dimensiones tras transformaciones: {df_transformado.shape}")
df_transformado.head(3)

Dimensiones tras transformaciones: (242631, 10)


,titulo,autor,paginas,calificacion,editorial,idioma,num_calificaciones,num_resenas,es_parte_de_serie,formato
0,La doctrina secreta. Vol I. Cosmogenesis (Cien...,Helena Petrovna Blavatsky,319.0,4.16,Editorial Kier,Spanish; Castilian,255.0,22.0,True,Paperback
3,Personality and Intellectual Competence,Tomas Chamorro-Premuzic,214.0,3.88,Psychology Press,English,8.0,0.0,False,Hardcover
7,Detik-Detik Diri Di Daerah Daif,Shahnon Ahmad,191.0,4.38,"Dewan Bahasa dan Pustaka, Kementerian Pendidik...",Malay,77.0,17.0,False,Paperback


In [4]:
# Instanciar la clase de reportes
reportes = Reportes()

# Imprimir el resumen en consola
reportes.imprimir_reporte(df_transformado)

# Guardar el reporte en archivo .txt
ruta_salida = reportes.guardar_reporte(
    df_transformado,
    "reportes/goodreads_books.txt"
)

print(f"Reporte exportado exitosamente en: {ruta_salida}")

REPORTE DEL DATAFRAME TRANSFORMADO
Filas: 242631
Columnas: 10
Duplicados: 89

Columnas y tipos:
- titulo: object
- autor: object
- paginas: float64
- calificacion: float64
- editorial: object
- idioma: object
- num_calificaciones: float64
- num_resenas: float64
- es_parte_de_serie: bool
- formato: object

Valores nulos por columna:
- titulo: 0
- autor: 0
- paginas: 0
- calificacion: 0
- editorial: 0
- idioma: 0
- num_calificaciones: 0
- num_resenas: 0
- es_parte_de_serie: 0
- formato: 0

Estadísticas numéricas:
            paginas  calificacion  num_calificaciones  num_resenas
count  2.426310e+05    242631.000        2.426310e+05   242631.000
mean   2.900041e+03         4.070        1.978588e+04      774.514
std    1.299554e+06         0.396        2.064168e+05     6397.337
min    0.000000e+00         3.500        1.000000e+00        0.000
25%    1.210000e+02         3.790        4.000000e+00        0.000
50%    2.240000e+02         4.000        2.200000e+01        3.000
75%    3.36000

Reporte exportado exitosamente en: reportes/goodreads_books.txt
